# 03: Econometric Diagnostic Testing & Robustness Analysis
**Repository:** `big5-player-valuation-econometrics`  
**Purpose:** Formal testing for heteroskedasticity (Breusch-Pagan), multicollinearity (VIF), residual normality, and sub-sample robustness checks.

## 1. Setup & Model Reconstruction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style="whitegrid")
df = pd.read_csv('data/processed/cleaned_players_data.csv')
df['log_market_value'] = np.log(df['market_value_eur'])
df['age_sq'] = df['age'] ** 2

# Fit Full Model
m_formula = 'log_market_value ~ age + age_sq + minutes_played + contract_years + C(position, Treatment(reference="Defender")) + xg_per90 + prog_passes_per90 + tackles_per90'
ols_model = smf.ols(m_formula, data=df).fit()
robust_model = smf.ols(m_formula, data=df).fit(cov_type='HC1')

## 2. Heteroskedasticity Testing (Breusch-Pagan Test)

In [ ]:
bp_test = het_breuschpagan(ols_model.resid, ols_model.model.exog)
bp_results = pd.DataFrame({
    'Metric': ['LM Statistic', 'LM p-value', 'F Statistic', 'F p-value'],
    'Value': [bp_test[0], bp_test[1], bp_test[2], bp_test[3]]
})
print("=== BREUSCH-PAGAN TEST RESULTS ===")
print(bp_results)
if bp_test[1] < 0.05:
    print("\nDecision: Reject H0 of Homoskedasticity. Heteroskedasticity present -> HC1 Robust SE required.")

## 3. Multicollinearity Assessment (VIF)

In [ ]:
X = ols_model.model.exog
vif_df = pd.DataFrame({
    'Variable': ols_model.model.exog_names,
    'VIF': [variance_inflation_factor(X, i) for i in range(X.shape[1])]
})
print("=== VARIANCE INFLATION FACTOR (VIF) ===")
print(vif_df)
print(f"Max VIF (excl. intercept): {vif_df[vif_df['Variable'] != 'Intercept']['VIF'].max():.2f}")

## 4. Residual Diagnostics (Saved to PNG and PDF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Residuals vs Fitted
axes[0].scatter(robust_model.fittedvalues, robust_model.resid, alpha=0.3, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Fitted Values')
axes[0].set_xlabel('Fitted ln(Market Value)')
axes[0].set_ylabel('Residuals')

# Q-Q Plot
sm.qqplot(robust_model.resid, line='s', ax=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.savefig('figures/residual_diagnostics.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/residual_diagnostics.pdf', bbox_inches='tight')
plt.close()
print("Residual diagnostic plots saved in PNG and PDF.")

## 5. Export Summary Metrics Table

In [ ]:
# Export summary metrics for reporting
summary = df.groupby('position').agg(
    player_count=('market_value_eur', 'count'),
    mean_market_val_eur=('market_value_eur', 'mean'),
    median_market_val_eur=('market_value_eur', 'median'),
    mean_age=('age', 'mean'),
    mean_minutes=('minutes_played', 'mean')
).reset_index()

summary.to_csv('data/processed/metrics_summary.csv', index=False)
print("Saved metrics_summary.csv to data/processed/")